# MiroFish Integration Demo

This notebook demonstrates loading data from a MiroFish social simulation,
computing interaction-based affinity, and running theories against detected communities.

**Requirements:**
```bash
pip install cft[mirofish]  # adds pandas and networkx
```

In [ ]:
import numpy as np
from pathlib import Path
from cft import CFT, GFT, TST
from cft.integrations.mirofish import MiroFishAdapter
from cft.visualization import plot_groups, plot_affinity_matrix

## 1. Load Agent Profiles

MiroFish produces agent profiles with MBTI types, opinion vectors, and influence scores.
The adapter converts these into numeric feature vectors.

We'll use the sample fixture data included in the repo.

In [ ]:
# Point to the sample fixture data
data_dir = Path("../tests/fixtures")
adapter = MiroFishAdapter(data_dir)

agents = adapter.load_agents()
print(f"Loaded {len(agents)} agents")
print(f"Feature dimensions: {agents[0].features.shape[0]}")
print(f"\nFirst agent:")
print(f"  ID: {agents[0].id}")
print(f"  MBTI: {agents[0].metadata.get('mbti')}")
print(f"  Influence: {agents[0].metadata.get('influence')}")
print(f"  Features: {agents[0].features}")

## 2. Load Interaction Data

Interactions include follows, likes, reposts, and comments between agents.
Each has a timestamp for temporal analysis.

In [ ]:
interactions = adapter.load_interactions()
print(f"Loaded {len(interactions)} interactions")
print(f"\nAction types:")
print(interactions["action"].value_counts().to_string())
print(f"\nTime range: {interactions['timestamp'].min()} to {interactions['timestamp'].max()}")

## 3. Compute Interaction-Based Affinity

Unlike feature-based affinity (Euclidean distance), this uses actual social interactions.
Positive actions (follow, like, repost) increase affinity; negative comments decrease it.

In [ ]:
from cft.integrations.mirofish import DEFAULT_WEIGHTS
print("Default interaction weights:")
for action, w in DEFAULT_WEIGHTS.items():
    print(f"  {action:>15s}: {w:+.1f}")

In [ ]:
affinity = adapter.compute_affinity_matrix()
print(f"Affinity matrix shape: {affinity.shape}")
print(f"Range: [{affinity.min():.3f}, {affinity.max():.3f}]")

fig = plot_affinity_matrix(affinity)
fig

## 4. Detect Ground-Truth Communities

Using Louvain community detection on the interaction graph to establish ground truth.

In [ ]:
ground_truth = adapter.extract_ground_truth_groups()
print(f"Detected {len(ground_truth)} communities:")
for g in ground_truth:
    print(f"  Group {g.id}: agents {g.members}")

In [ ]:
fig = plot_groups(ground_truth, agents, title="Ground Truth Communities")
fig

## 5. Run Prediction Pipeline

The prediction pipeline loads data, runs theories with interaction-based affinity,
and scores them against the detected communities.

In [ ]:
result = adapter.prediction_pipeline(
    theory_configs={
        "CFT": {"class": CFT, "threshold": 0.3},
        "GFT": {"class": GFT, "k": 0.3, "sigma": 1.5},
        "TST": {"class": TST, "temperature": 0.5, "sweeps_per_step": 10},
    },
    t_max=5.0,
    dt=1.0,
)

print("Scores:")
for name, s in result["scores"].items():
    print(f"  {name}: PAS={s['pas']:.3f}")

print(f"\nRankings:")
for r in result["rankings"]:
    print(f"  #{r['rank']} {r['theory']}: PAS={r['score']:.3f}")

## 6. Temporal Analysis: Freeze-and-Predict

Split interactions at a time boundary:
- **Before** the freeze: used to compute affinity (training data)
- **After** the freeze: used for ground-truth communities (evaluation)

This tests whether theories can predict future group structure from past interactions.

In [ ]:
# Reload to reset state
adapter2 = MiroFishAdapter(data_dir)
adapter2.load_agents()
adapter2.load_interactions()

result_temporal = adapter2.prediction_pipeline(
    theory_configs={
        "CFT": {"class": CFT, "threshold": 0.3},
        "TST": {"class": TST, "temperature": 0.5, "sweeps_per_step": 10},
    },
    t_freeze="2024-01-02T00:00:00",
    t_max=5.0,
    dt=1.0,
)

print("Temporal split results (train on Jan 1, evaluate on Jan 2+):")
for r in result_temporal["rankings"]:
    print(f"  #{r['rank']} {r['theory']}: PAS={r['score']:.3f}")

## Using Your Own MiroFish Data

To use with real MiroFish output, point the adapter at your simulation directory:

```python
adapter = MiroFishAdapter("/path/to/mirofish/simulation")
agents = adapter.load_agents()  # reads profiles.jsonl
interactions = adapter.load_interactions()  # reads actions.jsonl
```

Expected file formats:

**profiles.jsonl** - one JSON object per line:
```json
{"id": 0, "mbti": "ENFP", "opinions": [0.8, 0.3, -0.5], "influence": 0.9}
```

**actions.jsonl** - one JSON object per line:
```json
{"timestamp": "2024-01-01T10:00:00", "agent_i": 0, "agent_j": 1, "action": "follow"}
```

Action types: `follow`, `like`, `repost`, `pos_comment`, `neg_comment`